In [17]:
%pip install audioop-lts

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
from pydub import AudioSegment
from pydub.silence import split_on_silence

fichiers = ["mots1.m4a", "mots2.m4a"]

LISTE_MOTS = [
    "tournegauche",
    "demitour",
    "autorisation",
    "actionner",
    "recule",
    "allume",
    "statut",
    "danger",
    "stop",
    "vite",
    "mode",
    "bas",
    "hausse"
]

NB_ATTENDU_PAR_FICHIER = len(LISTE_MOTS) * 5  # 13 * 5 = 65

for k, fichier in enumerate(fichiers):
    print(f"\n{"="*40}")
    print(f"🎬 Traitement du fichier : {fichier}")
    print(f"{"="*40}")
    
    if not os.path.exists(fichier):
        print(f"❌ Erreur : Le fichier {fichier} n'existe pas dans le dossier.")
        continue

    print("Chargement de l'audio...")
    gros_audio = AudioSegment.from_file(fichier, format="m4a")
    gros_audio = gros_audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)

    # --- BOUCLE DE CALIBRATION AUTOMATIQUE PAR FICHIER ---
    morceaux = []
    print("🔍 Recherche du meilleur réglage de silence...")
    
    for seuil in range(-50, -15, 10):
        test_morceaux = split_on_silence(
            gros_audio,
            min_silence_len=800,  # Tolérance pour les pauses un peu plus courtes
            silence_thresh=seuil,
            keep_silence=250
        )
        
        if len(test_morceaux) == NB_ATTENDU_PAR_FICHIER:
            morceaux = test_morceaux
            print(f"🎯 Trouvé ! Seuil idéal pour {fichier} : {seuil} dB ({len(morceaux)} morceaux).")
            break
            
    # Si la boucle finit sans trouver pile 65
    if len(morceaux) != NB_ATTENDU_PAR_FICHIER:
        print(f"❌ Alerte : Impossible de trouver pile {NB_ATTENDU_PAR_FICHIER} morceaux pour {fichier}.")
        print(f"Dernier essai a donné {len(test_morceaux)} morceaux. Passage au fichier suivant.")
        continue

    # --- SAUVEGARDE DES MORCEAUX VALIDÉS ---
    print("💾 Tri et sauvegarde des fichiers...")
    for i, morceau in enumerate(morceaux):
        # Ta formule parfaite pour la continuité des sessions
        num_session = k * 5 + (i // len(LISTE_MOTS)) + 1
        
        index_mot = i % len(LISTE_MOTS)
        nom_mot = LISTE_MOTS[index_mot]
        
        dossier_mot = os.path.join("sons_ia", nom_mot)
        os.makedirs(dossier_mot, exist_ok=True)
        
        nom_fichier = f"{dossier_mot}/{nom_mot}_session_{num_session}.wav"
        morceau.export(nom_fichier, format="wav")

print("\n🎉 Tout est terminé ! Vérifie ton dossier 'sons_ia' pour voir tes dossiers par mots.")


🎬 Traitement du fichier : mots1.m4a
Chargement de l'audio...
🔍 Recherche du meilleur réglage de silence...
🎯 Trouvé ! Seuil idéal pour mots1.m4a : -30 dB (65 morceaux).
💾 Tri et sauvegarde des fichiers...

🎬 Traitement du fichier : mots2.m4a
Chargement de l'audio...
🔍 Recherche du meilleur réglage de silence...
🎯 Trouvé ! Seuil idéal pour mots2.m4a : -40 dB (65 morceaux).
💾 Tri et sauvegarde des fichiers...

🎉 Tout est terminé ! Vérifie ton dossier 'sons_ia' pour voir tes dossiers par mots.
